# Merging with `pd.merge()` (Inner, Outer, Left, Right Joins)

While concatenation simply glues DataFrames together, **merging** allows you to link tables together based on a **shared key column** (similar to a SQL `JOIN` operation).

For example, if you have a DataFrame containing Olympic Athlete bios (`athlete_id`, `name`, `born_country`) and another DataFrame containing raw event results (`athlete_id`, `event`, `medal`), you can merge them on the common `athlete_id` column to find out which athlete won which medal.

There are four primary ways to merge DataFrames, known as "joins":
1.  **Inner Join (`how='inner'`)**: Returns only the rows where the shared key exists in **both** DataFrames (Default behavior).
2.  **Full Outer Join (`how='outer'`)**: Returns all rows from both DataFrames, aligning matching keys and filling non-matching rows with `NaN`.
3.  **Left Join (`how='left'`)**: Keeps **all** rows from the left DataFrame, and pulls matching rows from the right DataFrame. Non-matching right rows are filled with `NaN`.
4.  **Right Join (`how='right'`)**: Keeps **all** rows from the right DataFrame, and pulls matching rows from the left DataFrame.

### Clear Explanation & Real-World Analogy
Think of two lists:
*   **List A (Left)**: Your friends' names and their favorite programming languages.
*   **List B (Right)**: Your friends' names and their birthdays.

*   **Inner Join**: You want to send programming-themed birthday cards. You can *only* do this for friends who are on **both** lists.
*   **Left Join**: You want to keep your entire list of friends (List A). If their birthday is on List B, you add it; if not, you leave it blank (`NaN`).
*   **Outer Join**: You combine both lists completely. If a friend has only a favorite language or only a birthday, they are still on the final list with a blank space for the missing info.

### Code Examples

Let's create two DataFrames with a common key column: `'EmployeeID'`.

In [1]:
import pandas as pd

# Left DataFrame: Roles
employees = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 104],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Role': ['TypeScript Dev', 'Python Dev', 'Data Analyst', 'UI Designer']
})

# Right DataFrame: Salaries (Notice EmployeeID 105 is new, and 104 is missing)
salaries = pd.DataFrame({
    'EmployeeID': [101, 102, 103, 105],
    'Salary': [85000, 95000, 90000, 120000]
})

#### Inner Join (`how='inner'`) - Default
Keeps only matching Employee IDs (`101, 102, 103`):

In [2]:
inner_merged = pd.merge(employees, salaries, on='EmployeeID', how='inner')
print(inner_merged)

   EmployeeID     Name            Role  Salary
0         101    Alice  TypeScript Dev   85000
1         102      Bob      Python Dev   95000
2         103  Charlie    Data Analyst   90000


#### Left Join (`how='left'`)
Keeps all employees from the left table (`Alice, Bob, Charlie, David`). David has no salary info in the right table, so his salary is `NaN`:


In [3]:
left_merged = pd.merge(employees, salaries, on='EmployeeID', how='left')
print(left_merged)

   EmployeeID     Name            Role   Salary
0         101    Alice  TypeScript Dev  85000.0
1         102      Bob      Python Dev  95000.0
2         103  Charlie    Data Analyst  90000.0
3         104    David     UI Designer      NaN


#### Full Outer Join (`how='outer'`)
Keeps everyone. David (104) has `NaN` salary, and Employee 105 has `NaN` Name and Role:

In [4]:
outer_merged = pd.merge(employees, salaries, on='EmployeeID', how='outer')
print(outer_merged)

   EmployeeID     Name            Role    Salary
0         101    Alice  TypeScript Dev   85000.0
1         102      Bob      Python Dev   95000.0
2         103  Charlie    Data Analyst   90000.0
3         104    David     UI Designer       NaN
4         105      NaN             NaN  120000.0



#### Joining on Different Column Names (`left_on` & `right_on`)
If the column names are different in each table (e.g., `'EmployeeID'` vs. `'ID'`), use `left_on` and `right_on` instead of `on`:

In [5]:
salaries_alt = pd.DataFrame({
    'ID': [101, 102],
    'Salary': [85000, 95000]
})

diff_key_merge = pd.merge(employees, salaries_alt, left_on='EmployeeID', right_on='ID', how='inner')
print(diff_key_merge)

   EmployeeID   Name            Role   ID  Salary
0         101  Alice  TypeScript Dev  101   85000
1         102    Bob      Python Dev  102   95000


### Common Pitfalls to Avoid
1.  **Duplicate Columns with Suffixes**: If the left and right DataFrames contain identical column names that are *not* used as keys (e.g., both have a `'Name'` column), Pandas will automatically append `_x` and `_y` suffixes to distinguish them. Use the `suffixes` parameter to name them cleanly, like `suffixes=('_left', '_right')`.
2.  **Duplicating Rows**: If your key column has duplicate values in both tables, the merge operation will match *every* occurrence of the key in the left table with *every* occurrence in the right table, leading to multiplied rows. Always inspect the shape of your DataFrame before and after a merge.

#### Exercise 1 (Medium)
We have a DataFrame of programming languages and their frameworks:
```python
languages = pd.DataFrame({
    'Lang': ['TypeScript', 'Python', 'JavaScript'],
    'Framework': ['Angular', 'Django', 'React']
})
```
Merge the `employees` DataFrame with the `languages` DataFrame on the appropriate columns to attach framework skills to each employee. Since some languages or employees might not match, perform a **left join** so no employees are dropped from the final output.



In [6]:
languages = pd.DataFrame({
    'Lang': ['TypeScript', 'Python', 'JavaScript'],
    'Framework': ['Angular', 'Django', 'React']
})

# Merge where 'Language' (from languages) matches 'Role' (Wait, Role is 'TypeScript Dev', 'Python Dev')
# Let's clean up and match correctly!
# Let's map languages 'Lang' to 'Role' by checking substrings or direct matching.
# For simplicity, let's match on the language columns:
# We'll first add a clean 'Language' column to the employees table, then merge.

employees['Language'] = ['TypeScript', 'Python', 'JavaScript', 'TypeScript']

completed_merge = pd.merge(employees, languages, left_on='Language', right_on='Lang', how='left')
print(completed_merge)

   EmployeeID     Name            Role    Language        Lang Framework
0         101    Alice  TypeScript Dev  TypeScript  TypeScript   Angular
1         102      Bob      Python Dev      Python      Python    Django
2         103  Charlie    Data Analyst  JavaScript  JavaScript     React
3         104    David     UI Designer  TypeScript  TypeScript   Angular


*Explanation:* By adding a standardized `'Language'` column, we can easily join it to the `'Lang'` column in our skills lookup DataFrame . Using `how='left'` ensures all original employees stay in the merged DataFrame.
